## 6. Jupyter Preprocessing Pipeline — Exact Steps

### 6.1 Load and Inspect

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('perf_metrics.csv')
print(df.shape)           # expect (N, 42)
print(df.dtypes)
print(df['session_label'].value_counts())

(5461, 44)
timestamp_ns                int64
pid                         int64
cpu                         int64
comm                       object
ctx_switches                int64
voluntary_switches          int64
involuntary_switches        int64
cpu_migrations              int64
total_runtime_ns            int64
stall_ns                    int64
avg_stall_ns                int64
max_stall_ns                int64
latency_count               int64
avg_runq_ratio              int64
minor_faults                int64
major_faults                int64
kmalloc_count               int64
kfree_count                 int64
total_alloc_bytes           int64
total_free_bytes            int64
large_page_allocs           int64
syscall_count               int64
avg_syscall_latency_ns      int64
max_syscall_latency_ns      int64
read_count                  int64
write_count                 int64
read_bytes                  int64
write_bytes                 int64
mmap_count                  int64
fut

### 6.2 Drop Useless Rows

In [2]:
# Drop rows where the process had zero context switches
# (process was registered in the map but never actually ran)
df = df[df['ctx_switches'] > 0].copy()

# Drop kernel threads with pid=0 (idle process — always class 0, adds noise)
df = df[df['pid'] > 0].copy()

# Drop rows with zero latency samples (no wakeup events recorded)
# These processes ran but were never woken up (e.g. initial exec)
# They can't be reliably classified
df = df[df['latency_count'] > 0].copy()

print(f"After cleaning: {len(df)} rows")

After cleaning: 4573 rows


### 6.3 Derive the ML Target Label

In [3]:
# Primary label from avg_stall_ns (runqueue latency)
bins   = [0, 2e6, 10e6, 50e6, float('inf')]
labels = [0,   1,    2,    3]
df['y'] = pd.cut(df['avg_stall_ns'], bins=bins, labels=labels).astype(int)

print(df['y'].value_counts().sort_index())

y
0    4573
Name: count, dtype: int64


**Verify the distribution makes sense:**
- `idle_baseline` sessions should be 90%+ class 0
- `cpu_high` sessions should push the stressed processes (stress-ng workers)
  to class 2–3; kernel threads and idle processes still appear as class 0

In [4]:
# Cross-check: session label vs derived ML class
print(pd.crosstab(df['session_label'], df['y']))

y                 0
session_label      
idle_baseline  4573


If `cpu_high` shows almost no class 2–3, your stressor is not causing enough
contention. Re-run with more threads.

### 6.4 Feature Engineering

In [5]:
# Ratio features (more informative than raw counts)
df['involuntary_ratio'] = (
    df['involuntary_switches'] / df['ctx_switches'].clip(lower=1)
)
df['voluntary_ratio'] = (
    df['voluntary_switches'] / df['ctx_switches'].clip(lower=1)
)
df['read_write_ratio'] = (
    df['read_bytes'] / (df['write_bytes'] + 1)   # +1 avoids div-by-zero
)
df['alloc_pressure'] = (
    df['total_alloc_bytes'] / df['ctx_switches'].clip(lower=1)
)
df['runtime_per_switch'] = (
    df['total_runtime_ns'] / df['ctx_switches'].clip(lower=1)
)
df['lock_pressure'] = (
    df['mutex_contentions'] + df['rwsem_read_contentions'] +
    df['rwsem_write_contentions']
)

# Log-scale the highly skewed latency columns
# (tree models handle raw values fine, but linear models need this)
for col in ['stall_ns', 'avg_stall_ns', 'max_stall_ns',
            'total_runtime_ns', 'avg_syscall_latency_ns']:
    df[f'log_{col}'] = np.log1p(df[col])

### 6.5 Feature Selection — What to Keep

In [6]:
# Columns to DROP before training:
drop_cols = [
    'timestamp_ns',    # raw timestamp, not a feature
    'pid',             # process identity, not a pattern
    'cpu',             # CPU affinity, not a bottleneck signal
    'comm',            # process name — encode separately if you want it
    'session_label',   # text descriptor, not the ML target
    'y',               # this is your target, not a feature
]

# Optional: encode comm as a categorical if you want process-type signal
# df['comm_enc'] = df['comm'].astype('category').cat.codes

feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols]
y = df['y']

### 6.6 Train/Test Split

In [7]:
from sklearn.model_selection import train_test_split

# IMPORTANT: split by session, not by row
# If you split by row, train and test will have rows from the same session
# and the model will memorise session-level statistics, not generalise.

# Assign each row a session index based on session_label
df['session_id'] = df['session_label'].astype('category').cat.codes

from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['session_id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

ValueError: With n_samples=1, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

### 6.7 Model Training

Start with Random Forest. It handles class imbalance, is robust to unscaled
features, and gives feature importance for free.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',   # handles class imbalance
    n_jobs=-1,
    random_state=42
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=['Normal', 'Low', 'Medium', 'High']))
print(confusion_matrix(y_test, y_pred))